# Name That LLM: treino LoRA24, etapa 1 completa (~35 mil textos, 1 época)

Qwen3-8B-Base nas primeiras 24 camadas, LoRA r16, leitura em 8 profundidades, cabeça linear, em duas T4 com dados
paralelos (um processo por GPU, gradientes somados por all_reduce). Treina na amostra balanceada do pool público
(`crtproduction/name-that-llm-open-pool`, sem as linhas geradas pelo proxy do Kaggle depois da regra do host de 28/10) e mede nos 2.699 textos
oficiais, que nunca entram no treino. Entradas: modelo `qwen-lm/qwen-3/Transformers/8b-base/1`, dataset `crtproduction/name-that-llm-open-pool` e a competição. Internet desligada.
Código: `scripts/treino_lora24.py`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

SCRIPT = '"""Etapa 1 do LoRA24: Qwen3 nas primeiras 24 camadas, lido em várias profundidades, treinado no pool público e medido\nnos 2.699 textos oficiais, que nunca entram no treino (por isso a nota é uma medida limpa).\n\nArquitetura da receita pública: LoRA r16 em todas as lineares, média dos tokens em 8 profundidades (cada uma com\nLayerNorm), cabeça linear, gradient checkpointing. Perda: entropia cruzada com amostra do pool balanceada por classe.\nTreina em 1 ou mais GPUs com dados paralelos: cada GPU guarda uma cópia das camadas e os gradientes (só do LoRA e das\ncabeças, ~30 M parâmetros) são somados por all_reduce, com escala de perda dinâmica própria (fp16 na T4), de modo que\ntodas as GPUs decidem juntas pular ou aplicar o passo.\n\nUso: python scripts/treino_lora24.py   (sem RANK no ambiente, ele mesmo lança um processo por GPU; "processos" na config\nforça outro número). Local (modelo minúsculo, CPU): TREINO_CFG=\'{"base": "...", "pool": "...", "oficial": "...",\n"processos": 2}\' python scripts/treino_lora24.py\nNo Kaggle: notebook gerado por scripts/montar_notebook_treino_lora24.py (internet desligada).\n"""\n\nimport json\nimport math\nimport os\nimport subprocess\nimport sys\nimport time\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nfrom torch import nn\n\nCFG = {\n    "base": "",  # pasta do modelo; vazio = procurar em /kaggle/input pelo base_spec\n    "base_spec": {"model_type": "qwen3", "hidden_size": 4096, "num_hidden_layers": 36},\n    "tokenizer": "",  # vazio = o da base\n    "keep": 24,\n    "layers": [2, 4, 6, 8, 12, 16, 20, 24],\n    "r": 16,\n    "max_len": 384,\n    "pool": "",  # vazio = procurar ntl_open_pool.parquet em /kaggle/input\n    "oficial": "",  # vazio = procurar train.csv em /kaggle/input\n    "n_pool_por_classe": 1944,  # 18 x 1944 = 34.992 textos\n    "excluir_proxy_pos_regra": True,  # linhas geradas pelo proxy do Kaggle depois da regra do host (28/10)\n    "micro_lote": 8,  # por GPU\n    "acumular": 1,\n    "epocas": 1,\n    "lr_nucleo": 1e-4,\n    "lr_cabeca": 1e-3,\n    "aquecimento_frac": 0.10,\n    "escala_inicial": 1024.0,  # escala da perda em fp16; em fp32 não se usa\n    "max_passos": 0,  # 0 = até acabar as épocas\n    "avaliar_cada": 500,  # passos; 0 desliga\n    "avaliar_n": 400,  # textos oficiais nas avaliações intermediárias\n    "avaliar_final_n": 0,  # 0 = todos os 2.699 oficiais\n    "salvar_cada": 500,\n    "tempo_maximo_s": 10800,\n    "saida": "/kaggle/working",\n    "backend": "",  # vazio = nccl com CUDA, gloo sem\n    "timeout_min": 30,\n    "processos": 0,  # 0 = um por GPU visível (ou 1 sem GPU)\n    "perfil_passos": False,  # mede tempos por passo (forward/backward, espera entre GPUs, all_reduce); só no check\n}\nCFG_JSON = "{}"  # o gerador do notebook troca esta linha\nCFG.update(json.loads(os.environ.get("TREINO_CFG", CFG_JSON)))\n\nRANK = int(os.environ.get("RANK", 0))\nMUNDO = int(os.environ.get("WORLD_SIZE", 1))\nLOCAL = int(os.environ.get("LOCAL_RANK", 0))\nT0 = time.time()\n\n\ndef log(*args):\n    if RANK == 0:\n        print(f"[{time.time() - T0:6.0f}s]", *args, flush=True)\n\n\ndef preparar_ambiente():\n    """O peft do Kaggle exige torchao >= 0.16 e a imagem traz o 0.10; sem o torchao o peft simplesmente não o usa."""\n    import importlib.metadata as md\n\n    try:\n        versao = md.version("torchao")\n    except md.PackageNotFoundError:\n        return\n    if tuple(int(x) for x in versao.split(".")[:2]) < (0, 16):\n        log(f"torchao {versao} incompatível com o peft: removendo")\n        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)\n\n\ndef achar(padrao: str) -> str:\n    achados = sorted(Path("/kaggle/input").rglob(padrao))\n    if not achados:\n        raise FileNotFoundError(f"{padrao} não encontrado em /kaggle/input")\n    return str(achados[0])\n\n\ndef achar_base() -> str:\n    if CFG["base"]:\n        return CFG["base"]\n    spec = CFG["base_spec"]\n    for cfg in sorted(Path("/kaggle/input").rglob("config.json")):\n        try:\n            c = json.loads(cfg.read_text())\n        except Exception:\n            continue\n        if all(c.get(k) == v for k, v in spec.items()):\n            return str(cfg.parent)\n    raise FileNotFoundError(f"modelo {spec} não encontrado em /kaggle/input")\n\n\ndef normalizar(texto: str) -> str:\n    return " ".join(texto.split()).lower()\n\n\ndef carregar_dados(tok):\n    oficial = pd.read_csv(CFG["oficial"] or achar("train.csv"), keep_default_na=False)\n    classes = sorted(oficial.model.unique())\n    assert len(classes) == 18, f"{len(classes)} classes nos oficiais"\n    pool = pd.read_parquet(CFG["pool"] or achar("ntl_open_pool.parquet"),\n                           columns=["text", "model", "n_words", "flag_kb_proxy_after_ruling"])\n    n0 = len(pool)\n    pool = pool[pool.n_words.between(30, 300)]\n    if CFG["excluir_proxy_pos_regra"]:\n        pool = pool[~pool.flag_kb_proxy_after_ruling.astype(bool)]\n    oficiais = {normalizar(t) for t in oficial.text}  # nenhum texto oficial pode estar no treino\n    pool = pool[~pool.text.map(normalizar).isin(oficiais)].drop_duplicates("text")\n    assert set(pool.model) <= set(classes), f"classes do pool fora das oficiais: {set(pool.model) - set(classes)}"\n    amostra = pd.concat([g.sample(min(len(g), CFG["n_pool_por_classe"]), random_state=0) for _, g in pool.groupby("model")])\n    amostra = amostra.sample(frac=1, random_state=0).reset_index(drop=True)\n    log(f"pool: {n0} linhas, {len(pool)} após filtros, {len(amostra)} na amostra balanceada "\n        f"({amostra.model.value_counts().min()} a {amostra.model.value_counts().max()} por classe)")\n\n    def tokenizar(textos):\n        bruto = tok([t.strip() for t in textos], add_special_tokens=False)["input_ids"]\n        return [x[: CFG["max_len"] - 1] + [tok.eos_token_id] for x in bruto]\n\n    indice = {c: i for i, c in enumerate(classes)}\n    return (tokenizar(amostra.text), amostra.model.map(indice).to_numpy(),\n            tokenizar(oficial.text), oficial.model.map(indice).to_numpy(), classes)\n\n\nclass Leitor(nn.Module):\n    """Camadas do Qwen3 com LoRA, média dos tokens em várias profundidades (cada uma com LayerNorm) e cabeça linear."""\n\n    def __init__(self, trunk, layers, hidden, n_classes):\n        super().__init__()\n        self.trunk, self.layers = trunk, layers\n        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in layers])\n        self.head = nn.Linear(hidden * len(layers), n_classes)\n\n    def forward(self, ids, mask):\n        hs = self.trunk(input_ids=ids, attention_mask=mask, output_hidden_states=True, use_cache=False).hidden_states\n        assert len(hs) == CFG["keep"] + 1, f"hidden_states com {len(hs)} entradas, esperado {CFG[\'keep\'] + 1}"\n        m = mask.float().clone()\n        m[:, 0] = 0  # o primeiro token é um "attention sink": fora da média\n        m = m.unsqueeze(-1)\n        den = m.sum(1).clamp(min=1)\n        z = torch.cat([nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(self.layers, self.norms)], -1)\n        return self.head(z)\n\n\ndef montar_modelo(base, dispositivo, tipo, n_classes):\n    from peft import LoraConfig, TaskType, get_peft_model\n    from transformers import AutoConfig, AutoModel\n\n    mc = AutoConfig.from_pretrained(base)\n    mc.num_hidden_layers, mc.use_cache = CFG["keep"], False\n    if getattr(mc, "layer_types", None) is not None:\n        mc.layer_types = mc.layer_types[: CFG["keep"]]\n    if dispositivo.startswith("cuda"):\n        trunk = AutoModel.from_pretrained(base, config=mc, dtype=tipo, device_map={"": LOCAL})\n    else:\n        trunk = AutoModel.from_pretrained(base, config=mc, dtype=tipo).to(dispositivo)\n    trunk.norm = nn.Identity()  # a receita pública lê a camada 24 sem a normalização final\n    trunk.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    trunk.enable_input_require_grads()\n    lora = LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, r=CFG["r"], lora_alpha=2 * CFG["r"], lora_dropout=0.05,\n                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])\n    trunk = get_peft_model(trunk, lora)\n    for p in trunk.parameters():\n        if p.requires_grad:\n            p.data = p.data.float()  # só o adaptador treina, em fp32\n    leitor = Leitor(trunk, CFG["layers"], mc.hidden_size, n_classes)\n    leitor.norms.to(dispositivo)\n    leitor.head.to(dispositivo)\n    treinaveis = sum(p.numel() for p in leitor.parameters() if p.requires_grad)\n    log(f"modelo com {CFG[\'keep\']} camadas em {dispositivo}/{tipo}, treináveis {treinaveis / 1e6:.1f} M, {MUNDO} GPU(s)")\n    return leitor\n\n\ndef parametros_treinaveis(leitor):\n    nucleo = [p for n, p in leitor.named_parameters() if p.requires_grad and n.startswith("trunk")]\n    cabeca = [p for n, p in leitor.named_parameters() if p.requires_grad and not n.startswith("trunk")]\n    return nucleo, cabeca\n\n\ndef lotes(ids, tamanho, rng, mundo=1):\n    """Lista global de lotes (iguais em todas as GPUs): tamanhos parecidos juntos, ordem embaralhada.\n    Com várias GPUs, cada grupo de `mundo` lotes seguidos (um por GPU, no mesmo passo) tem comprimentos parecidos:\n    sem isso cada passo espera o lote mais longo dos sorteados (medido: +26% de tempo com 2 GPUs)."""\n    ordem = rng.permutation(len(ids))\n    saida = []\n    for i in range(0, len(ids), tamanho * 50):\n        bloco = sorted(ordem[i : i + tamanho * 50], key=lambda j: len(ids[j]))\n        saida += [bloco[k : k + tamanho] for k in range(0, len(bloco), tamanho)]\n    saida = [b for b in saida if len(b) == tamanho]\n    if mundo == 1:\n        rng.shuffle(saida)\n        return saida\n    saida.sort(key=lambda b: max(len(ids[i]) for i in b))\n    grupos = [saida[k : k + mundo] for k in range(0, len(saida) - mundo + 1, mundo)]\n    for grupo in grupos:\n        rng.shuffle(grupo)  # a mesma GPU não fica sempre com o lote mais curto do par\n    rng.shuffle(grupos)\n    return [b for grupo in grupos for b in grupo]\n\n\ndef sincronizar(dispositivo):\n    if dispositivo.startswith("cuda"):\n        torch.cuda.synchronize()\n    elif dispositivo == "mps":\n        torch.mps.synchronize()\n\n\ndef montar_lote(ids, indices, pad, dispositivo):\n    maior = max(len(ids[i]) for i in indices)\n    x = torch.full((len(indices), maior), pad, dtype=torch.long)\n    m = torch.zeros((len(indices), maior), dtype=torch.long)\n    for k, i in enumerate(indices):\n        x[k, : len(ids[i])] = torch.tensor(ids[i])\n        m[k, : len(ids[i])] = 1\n    return x.to(dispositivo), m.to(dispositivo)\n\n\n@torch.no_grad()\ndef prever(leitor, ids, pad, dispositivo, tipo, tamanho=16):\n    """Logits de todos os textos; cada GPU faz uma fatia (i % mundo == rank) e as fatias são reunidas."""\n    leitor.eval()\n    if dispositivo.startswith("cuda"):\n        torch.cuda.empty_cache()\n    n = len(ids)\n    meus = np.arange(n)[RANK::MUNDO]\n    k = math.ceil(n / MUNDO)\n    local = torch.zeros((k, 18), device=dispositivo)\n    ordem = sorted(range(len(meus)), key=lambda j: len(ids[meus[j]]))\n    tipo_disp = dispositivo.split(":")[0]\n    for a in range(0, len(ordem), tamanho):\n        pos = ordem[a : a + tamanho]\n        x, m = montar_lote(ids, [meus[j] for j in pos], pad, dispositivo)\n        with torch.autocast(device_type=tipo_disp, dtype=tipo, enabled=tipo != torch.float32):\n            local[pos] = leitor(x, m).float()\n    if MUNDO > 1:\n        partes = [torch.zeros_like(local) for _ in range(MUNDO)]\n        dist.all_gather(partes, local)\n    else:\n        partes = [local]\n    saida = np.zeros((n, 18), dtype=np.float32)\n    for r in range(MUNDO):\n        quantos = len(np.arange(n)[r::MUNDO])\n        saida[np.arange(n)[r::MUNDO]] = partes[r][:quantos].cpu().numpy()\n    leitor.train()\n    return saida\n\n\ndef nota(y, logits):\n    y = np.asarray(y)\n    previsto = logits.argmax(1)\n    return float(np.mean([(previsto[y == c] == c).mean() for c in np.unique(y)]))\n\n\ndef passo_otimizador(todos, otimizador, escala):\n    """Soma os gradientes entre as GPUs, tira a escala, corta a norma e aplica. Retorna (aplicou, norma).\n    Todas as GPUs recebem o mesmo vetor somado, então decidem igual: se uma teve inf/nan, todas pulam."""\n    grads = [p.grad if p.grad is not None else torch.zeros_like(p) for p in todos]\n    plano = torch.cat([g.reshape(-1) for g in grads])\n    if MUNDO > 1:\n        dist.all_reduce(plano)\n    plano /= escala * MUNDO\n    norma = plano.norm()\n    if not bool(torch.isfinite(norma)):\n        return False, float("nan")\n    plano *= min(1.0, 1.0 / (float(norma) + 1e-6))\n    inicio = 0\n    for p, g in zip(todos, grads):\n        p.grad = plano[inicio : inicio + g.numel()].view_as(g).clone()\n        inicio += g.numel()\n    otimizador.step()\n    return True, float(norma)\n\n\ndef treinar(leitor, ids, y, ids_of, y_of, pad, dispositivo, tipo, resultado):\n    nucleo, cabeca = parametros_treinaveis(leitor)\n    todos = nucleo + cabeca\n    otimizador = torch.optim.AdamW([{"params": nucleo, "lr": CFG["lr_nucleo"]}, {"params": cabeca, "lr": CFG["lr_cabeca"]}],\n                                   weight_decay=0.0)\n    base_lr = [CFG["lr_nucleo"], CFG["lr_cabeca"]]\n    fp16 = tipo == torch.float16\n    escala = float(CFG["escala_inicial"]) if fp16 else 1.0\n    por_passo = MUNDO * CFG["acumular"]\n    passos_epoca = len(lotes(ids, CFG["micro_lote"], np.random.default_rng(1), MUNDO)) // por_passo\n    total = passos_epoca * CFG["epocas"]\n    if CFG["max_passos"]:\n        total = min(total, CFG["max_passos"])\n    aquec = max(1, int(CFG["aquecimento_frac"] * total))\n    tipo_disp = dispositivo.split(":")[0]\n    yt = torch.as_tensor(y, dtype=torch.long)\n    avaliar_idx = np.random.default_rng(2).permutation(len(ids_of))[: CFG["avaliar_n"]]\n    ids_av, y_av = [ids_of[i] for i in avaliar_idx], y_of[avaliar_idx]\n    resultado.update(passos_total_previsto=total, passos_por_epoca=passos_epoca, avaliacoes=[], historico_perda=[])\n    log(f"{len(ids)} textos, {passos_epoca} passos por época, {total} passos no total, micro-lote {CFG[\'micro_lote\']} por GPU")\n\n    leitor.train()\n    passo, tokens, perda_media, pulados, bons_seguidos = 0, 0, None, 0, 0\n    t_aval = 0.0  # tempo gasto avaliando: fora da velocidade de treino\n    perfil, acum = CFG["perfil_passos"], {"passos": 0, "fwbw_s": 0.0, "espera_s": 0.0, "all_reduce_e_otimizador_s": 0.0}\n    t_ini = time.time()\n    parar = False\n    for epoca in range(CFG["epocas"]):\n        fila = lotes(ids, CFG["micro_lote"], np.random.default_rng(1 + epoca), MUNDO)\n        for g in range(passos_epoca):\n            if passo >= total or parar:\n                break\n            fator = min(1.0, (passo + 1) / aquec) * 0.5 * (1 + math.cos(math.pi * passo / max(1, total)))\n            for grupo, lr in zip(otimizador.param_groups, base_lr):\n                grupo["lr"] = lr * fator\n            t_a = time.time()\n            for a in range(CFG["acumular"]):\n                indices = fila[(g * CFG["acumular"] + a) * MUNDO + RANK]\n                x, m = montar_lote(ids, indices, pad, dispositivo)\n                with torch.autocast(device_type=tipo_disp, dtype=tipo, enabled=tipo != torch.float32):\n                    logits = leitor(x, m)\n                perda = torch.nn.functional.cross_entropy(logits.float(), yt[indices].to(dispositivo))\n                (perda * escala / CFG["acumular"]).backward()\n                tokens += int(m.sum())\n                v = float(perda)\n                if math.isfinite(v):\n                    perda_media = v if perda_media is None else 0.98 * perda_media + 0.02 * v\n            if perfil:  # separa: forward/backward, espera pela outra GPU (desbalanceamento) e all_reduce + otimizador\n                sincronizar(dispositivo)\n                t_b = time.time()\n                if MUNDO > 1:\n                    dist.barrier()\n                    sincronizar(dispositivo)\n                t_c = time.time()\n            aplicou, _ = passo_otimizador(todos, otimizador, escala)\n            otimizador.zero_grad(set_to_none=True)\n            if perfil:\n                sincronizar(dispositivo)\n                if passo >= 3:  # sem o aquecimento do CUDA\n                    acum["passos"] += 1\n                    acum["fwbw_s"] += t_b - t_a\n                    acum["espera_s"] += t_c - t_b\n                    acum["all_reduce_e_otimizador_s"] += time.time() - t_c\n            if aplicou:\n                bons_seguidos += 1\n                if fp16 and bons_seguidos >= 200 and escala < 2 ** 16:\n                    escala, bons_seguidos = escala * 2, 0\n            else:\n                pulados += 1\n                bons_seguidos = 0\n                if fp16:\n                    escala = max(1.0, escala / 2)\n                log(f"passo {passo}: gradiente não finito, passo pulado; escala {escala:g}")\n            passo += 1\n            if passo % 20 == 0:\n                dt = time.time() - t_ini - t_aval\n                resultado["historico_perda"].append([passo, perda_media])\n                eta = dt / passo * (total - passo)\n                log(f"passo {passo}/{total} perda {perda_media:.3f} {tokens * MUNDO / dt:.0f} tokens/s (todas as GPUs) "\n                    f"escala {escala:g} pulados {pulados} ETA {eta / 60:.0f} min")\n                if MUNDO > 1:  # as GPUs decidem juntas parar por tempo\n                    flag = torch.tensor([1.0 if time.time() - T0 > CFG["tempo_maximo_s"] else 0.0], device=dispositivo)\n                    dist.broadcast(flag, 0)\n                    parar = bool(flag.item())\n                else:\n                    parar = time.time() - T0 > CFG["tempo_maximo_s"]\n                if parar:\n                    log(f"tempo máximo atingido no passo {passo}")\n            if CFG["avaliar_cada"] and passo % CFG["avaliar_cada"] == 0:\n                t_e = time.time()\n                ba = nota(y_av, prever(leitor, ids_av, pad, dispositivo, tipo))\n                t_aval += time.time() - t_e\n                resultado["avaliacoes"].append({"passo": passo, "balanced_accuracy": ba, "textos": len(ids_av)})\n                log(f"AVALIAÇÃO passo {passo}: balanced accuracy {ba:.4f} em {len(ids_av)} textos oficiais")\n            if CFG["salvar_cada"] and passo % CFG["salvar_cada"] == 0:\n                salvar(leitor)\n        if passo >= total or parar:\n            break\n    dt = time.time() - t_ini - t_aval\n    resultado.update(passos=passo, passos_pulados=pulados, escala_final=escala, tempo_treino_s=dt, tempo_avaliacao_s=t_aval,\n                     tokens_por_s_todas_gpus=tokens * MUNDO / dt, perda_final=perda_media)\n    if perfil and acum["passos"]:\n        n = acum["passos"]\n        resultado["perfil_passos"] = {"passos_medidos": n, "fwbw_s": acum["fwbw_s"] / n, "espera_barreira_s": acum["espera_s"] / n,\n                                      "all_reduce_e_otimizador_s": acum["all_reduce_e_otimizador_s"] / n}\n        log("PERFIL por passo:", json.dumps({k: round(v, 3) for k, v in resultado["perfil_passos"].items()}))\n\n\ndef salvar(leitor):\n    if RANK != 0:\n        return\n    saida = Path(CFG["saida"])\n    leitor.trunk.save_pretrained(saida / "adaptador")\n    torch.save({"norms": leitor.norms.state_dict(), "head": leitor.head.state_dict()}, saida / "cabecas.pt")\n\n\ndef main():\n    from transformers import AutoTokenizer\n\n    saida = Path(CFG["saida"])\n    saida.mkdir(parents=True, exist_ok=True)\n    cuda = torch.cuda.is_available()\n    if cuda:\n        dispositivo, tipo = f"cuda:{LOCAL}", torch.float16  # T4 não tem bf16 nativo\n        torch.cuda.set_device(LOCAL)\n    elif MUNDO == 1 and torch.backends.mps.is_available() and os.environ.get("TREINO_MPS"):\n        dispositivo, tipo = "mps", torch.bfloat16\n    else:\n        dispositivo, tipo = "cpu", torch.float32\n    if MUNDO > 1:\n        backend = CFG["backend"] or ("nccl" if cuda else "gloo")\n        dist.init_process_group(backend=backend, timeout=timedelta(minutes=CFG["timeout_min"]))\n        log(f"distribuído: backend {backend}, {MUNDO} processos")\n    if RANK == 0:\n        preparar_ambiente()\n    if MUNDO > 1:\n        dist.barrier()\n    torch.manual_seed(0)\n    np.random.seed(0)\n\n    base = achar_base()\n    tok = AutoTokenizer.from_pretrained(CFG["tokenizer"] or base)\n    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id\n    ids, y, ids_of, y_of, classes = carregar_dados(tok)\n    log(f"{np.mean([len(i) for i in ids]):.0f} tokens por texto do pool em média; {len(ids_of)} textos oficiais para avaliar")\n    t_carga = time.time()\n    leitor = montar_modelo(base, dispositivo, tipo, len(classes))\n    if MUNDO > 1:  # mesmas cabeças e mesmo LoRA em todas as GPUs, mesmo que a inicialização divirja\n        for p in sum(parametros_treinaveis(leitor), []):\n            dist.broadcast(p.data, 0)\n    resultado = {"config": CFG, "classes": classes, "mundo": MUNDO, "carregamento_s": time.time() - t_carga,\n                 "dispositivo": dispositivo}\n    if cuda:\n        resultado["gpu"] = torch.cuda.get_device_name(LOCAL)\n\n    treinar(leitor, ids, y, ids_of, y_of, pad, dispositivo, tipo, resultado)\n\n    # as GPUs têm de terminar com os mesmos parâmetros treináveis\n    soma = torch.stack([p.detach().double().sum() for p in sum(parametros_treinaveis(leitor), [])]).sum().reshape(1)\n    somas = [torch.zeros_like(soma) for _ in range(MUNDO)]\n    if MUNDO > 1:\n        dist.all_gather(somas, soma)\n    else:\n        somas[0] = soma\n    resultado["diferenca_params_entre_gpus"] = float(max(s.item() for s in somas) - min(s.item() for s in somas))\n    salvar(leitor)\n\n    idx = np.arange(len(ids_of))[: CFG["avaliar_final_n"] or len(ids_of)]\n    logits = prever(leitor, [ids_of[i] for i in idx], pad, dispositivo, tipo)\n    yv = y_of[idx]\n    previsto = logits.argmax(1)\n    familia = np.array([c.split("-")[0] for c in classes])\n    resultado["final"] = {\n        "textos": int(len(idx)), "balanced_accuracy": nota(yv, logits),\n        "acuracia_familia": float((familia[previsto] == familia[yv]).mean()),\n        "recall_por_classe": {classes[c]: float((previsto[yv == c] == c).mean()) for c in np.unique(yv)},\n    }\n    if RANK == 0:\n        np.save(saida / "logits_oficial.npy", logits)\n        np.save(saida / "y_oficial.npy", yv)\n        (saida / "resultado.json").write_text(json.dumps(resultado, indent=2))\n    log("FINAL balanced accuracy nos oficiais:", round(resultado["final"]["balanced_accuracy"], 4),\n        "| família:", round(resultado["final"]["acuracia_familia"], 4),\n        "| passos pulados:", resultado["passos_pulados"], "| diferença entre GPUs:", resultado["diferenca_params_entre_gpus"])\n    if MUNDO > 1:\n        dist.barrier()\n        dist.destroy_process_group()\n\n\ndef lancar() -> int:\n    """Lança um processo por GPU com o ambiente que o init_process_group espera (endereço fixo em 127.0.0.1: o\n    torchrun --standalone trava no macOS ao resolver o hostname). Se um processo cair, derruba os outros."""\n    import shutil\n    import socket\n\n    if CFG["processos"]:\n        n = CFG["processos"]\n    else:\n        listagem = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None\n        n = max(1, sum(1 for linha in (listagem.stdout.splitlines() if listagem else []) if linha.startswith("GPU ")))\n    with socket.socket() as s:\n        s.bind(("127.0.0.1", 0))\n        porta = s.getsockname()[1]\n    env = dict(os.environ, WORLD_SIZE=str(n), MASTER_ADDR="127.0.0.1", MASTER_PORT=str(porta), TREINO_CFG=json.dumps(CFG))\n    env.setdefault("GLOO_SOCKET_IFNAME", "lo0" if sys.platform == "darwin" else "lo")\n    print(f"lançando {n} processo(s)", flush=True)\n    procs = [subprocess.Popen([sys.executable, str(Path(__file__).resolve())],\n                              env=dict(env, RANK=str(i), LOCAL_RANK=str(i))) for i in range(n)]\n    while True:\n        codigos = [p.poll() for p in procs]\n        if all(c is not None for c in codigos):\n            break\n        if any(c not in (None, 0) for c in codigos):\n            print(f"um processo terminou com erro {codigos}: derrubando os outros", flush=True)\n            for p in procs:\n                if p.poll() is None:\n                    p.terminate()\n            break\n        time.sleep(2)\n    codigos = [p.wait() for p in procs]\n    print("códigos de saída:", codigos, flush=True)\n    return 0 if all(c == 0 for c in codigos) else 1\n\n\nif __name__ == "__main__":\n    if "RANK" in os.environ:\n        main()\n    else:\n        sys.exit(lancar())\n'
caminho = Path("/kaggle/working/treino_lora24.py")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(SCRIPT)
processo = subprocess.Popen([sys.executable, str(caminho)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linha in processo.stdout:
    print(linha, end="")
codigo = processo.wait()
print("código de saída:", codigo)
resultado = Path("/kaggle/working/resultado.json")
if resultado.exists():
    res = json.loads(resultado.read_text())
    print(json.dumps({k: v for k, v in res.items() if k not in ("config", "historico_perda")}, indent=2)[:6000])
assert codigo == 0
